In [ ]:
import pandas as pd
import numpy as np

In [ ]:
merged_df = pd.read_csv('merged_data.csv')

In [ ]:
print(f"Loaded merged data: {len(merged_df)} rows, {merged_df['RID'].nunique()} unique patients")

Loaded merged data: 4226 rows, 1274 unique patients


In [ ]:
#get meta columns
exclude_cols = ['RID', 'VISCODE', 'DX', 'DX_bl', 'Month']
metabolite_cols = [col for col in merged_df.columns if col not in exclude_cols]

print(f"Total metabolite columns: {len(metabolite_cols)}")
missing_counts = merged_df[metabolite_cols].isnull().sum(axis=1)
rows_with_missing = (missing_counts > 0).sum()

print(f"Rows with missing metabolite data: {rows_with_missing}")

Total metabolite columns: 104
Rows with missing metabolite data: 1


In [ ]:
print(f"Patients before cleaning: {len(merged_df)}")
merged_df = merged_df.dropna(subset=metabolite_cols)
print(f"Patients after dropping missing values: {len(merged_df)}")

Patients before cleaning: 4226
Patients after dropping missing values: 4225


In [ ]:
print("Available diagnosis labels:")
print(merged_df['DX'].value_counts())
print("\nAvailable VISCODE values:")
print(merged_df['VISCODE'].value_counts())

Available diagnosis labels:
DX
MCI         1821
CN          1260
Dementia    1144
Name: count, dtype: int64

Available VISCODE values:
VISCODE
bl      1179
m12     1047
m24      950
m36      200
m48      164
m60      156
m06      142
m72      133
m18      103
m84       93
m96       42
m108      11
m120       5
Name: count, dtype: int64


In [ ]:
#define progression order
progression_order = ['CN', 'EMCI', 'MCI', 'LMCI', 'Dementia']

def get_progression_score(dx):
    if pd.isna(dx) or dx not in progression_order:
        return -1
    return progression_order.index(dx)

In [ ]:
progression_data = []

for rid in merged_df['RID'].unique():
    patient_data = merged_df[merged_df['RID'] == rid].sort_values('Month')

    #get bl data
    baseline_row = patient_data[patient_data['VISCODE'] == 'bl']
    if len(baseline_row) == 0:
        continue  # skip patients without baseline

    baseline_row = baseline_row.iloc[0]
    baseline_dx = baseline_row['DX_bl']

    #track progression thru months
    visits = patient_data.sort_values('Month')
    max_progression_score = get_progression_score(baseline_dx)
    progression_dx = baseline_dx
    progression_month = baseline_row['Month']

    #identify most severe diagnosis + month occurred
    for _, visit in visits.iterrows():
        visit_score = get_progression_score(visit['DX'])
        if visit_score > max_progression_score:
            max_progression_score = visit_score
            progression_dx = visit['DX']
            progression_month = visit['Month']

    #create progression entry
    progression_entry = baseline_row.copy()
    progression_entry['BASELINE_DX'] = baseline_dx  # Keep original label
    progression_entry['FINAL_DX'] = progression_dx  # Keep original label
    progression_entry['PROGRESSION_MONTH'] = progression_month
    progression_entry['PROGRESSED'] = 1 if progression_dx != baseline_dx else 0
    progression_entry['TOTAL_VISITS'] = len(patient_data)

    progression_data.append(progression_entry)

In [ ]:
progression_df = pd.DataFrame(progression_data)
print(f"Progression dataset: {len(progression_df)} patients (baseline only)")

Progression dataset: 1179 patients (baseline only)


In [ ]:
print("\nBaseline diagnosis distribution:")
print(progression_df['BASELINE_DX'].value_counts())

print("\nFinal diagnosis distribution:")
print(progression_df['FINAL_DX'].value_counts())

print("\nProgression summary:")
print(f"Patients who progressed: {progression_df['PROGRESSED'].sum()}")
print(f"Patients who stayed stable: {(progression_df['PROGRESSED'] == 0).sum()}")


Baseline diagnosis distribution:
BASELINE_DX
LMCI        421
CN          350
EMCI        222
Dementia    186
Name: count, dtype: int64

Final diagnosis distribution:
FINAL_DX
Dementia    458
CN          276
MCI         253
LMCI        191
EMCI          1
Name: count, dtype: int64

Progression summary:
Patients who progressed: 525
Patients who stayed stable: 654


In [ ]:
print("\nProgression patterns:")
progression_patterns = progression_df.groupby(['BASELINE_DX', 'FINAL_DX']).size().reset_index(name='count')
for _, row in progression_patterns.iterrows():
    if row['BASELINE_DX'] != row['FINAL_DX']:
        print(f"{row['BASELINE_DX']} → {row['FINAL_DX']}: {row['count']} patients")


Progression patterns:
CN → Dementia: 17 patients
CN → MCI: 57 patients
EMCI → Dementia: 25 patients
EMCI → MCI: 196 patients
LMCI → Dementia: 230 patients


In [ ]:
print("\nAverage progression time (months):")
progressed_patients = progression_df[progression_df['PROGRESSED'] == 1]
if len(progressed_patients) > 0:
    avg_progression_time = progressed_patients.groupby(['BASELINE_DX', 'FINAL_DX'])['PROGRESSION_MONTH'].mean()
    for (baseline, final), avg_time in avg_progression_time.items():
        print(f"{baseline} → {final}: {avg_time:.1f} months")


Average progression time (months):
CN → Dementia: 73.1 months
CN → MCI: 46.6 months
EMCI → Dementia: 29.3 months
EMCI → MCI: 0.0 months
LMCI → Dementia: 26.5 months


In [ ]:
#drop month column since it's all 0s from baseline
progression_df = progression_df.drop(columns=['Month', 'DX',])

In [ ]:
progression_cols = ['RID', 'VISCODE', 'DX_bl', 'BASELINE_DX', 'FINAL_DX', 'PROGRESSION_MONTH', 'PROGRESSED', 'TOTAL_VISITS']
other_cols = [col for col in progression_df.columns if col not in progression_cols]
column_order = progression_cols + other_cols

progression_df = progression_df[column_order]

In [ ]:
list(progression_df.columns[:10])

['RID',
 'VISCODE',
 'DX_bl',
 'BASELINE_DX',
 'FINAL_DX',
 'PROGRESSION_MONTH',
 'PROGRESSED',
 'TOTAL_VISITS',
 'L_HISTIDINE',
 'L_LYSINE']

In [ ]:
progression_df.head(10)

,RID,VISCODE,DX_bl,BASELINE_DX,FINAL_DX,PROGRESSION_MONTH,PROGRESSED,TOTAL_VISITS,L_HISTIDINE,L_LYSINE,...,NORCA,NORDCA,TCA,TCDCA,TDCA,TUDCA,TAMCA,UCA,UDCA,BUDCA
0,3,bl,Dementia,Dementia,Dementia,0,0,3,99.4952,261.459,...,16.97580,2.93506,55.71210,154.11700,175.79600,1.7,11.3,0.818550,15.6000,25.1
3,4,bl,LMCI,LMCI,LMCI,0,0,2,78.1877,208.347,...,4.12863,2.05665,15.22970,97.38330,26.79270,2.1,1.6,0.225801,24.5000,216.8
5,5,bl,CN,CN,CN,0,0,3,95.3085,185.013,...,11.09640,1.30000,349.33300,501.46500,455.26400,11.1,13.3,1.239740,86.8000,224.3
8,6,bl,LMCI,LMCI,LMCI,0,0,3,89.7745,297.523,...,3.43190,3.59681,16.82620,75.07010,13.34210,1.6,1.3,0.991451,13.1000,19.1
10,7,bl,Dementia,Dementia,Dementia,0,0,3,115.9300,252.900,...,20.98920,4.03903,14.28970,24.61530,6.87759,1.1,2.0,23.490600,79.6000,342.9
11,10,bl,Dementia,Dementia,Dementia,0,0,3,82.5696,286.193,...,3.79158,3.18804,7.46819,7.57962,2.13568,0.5,4.9,0.474664,11.3000,19.1
14,14,bl,CN,CN,CN,0,0,3,93.0403,190.911,...,8.36856,2.58393,19.29850,57.32890,44.46280,1.6,1.6,0.577279,69.4000,179.2
17,15,bl,CN,CN,MCI,72,1,3,85.9260,119.922,...,13.87970,3.83974,12.96690,25.06050,28.71790,0.6,6.6,0.541429,26.3000,71.4
18,16,bl,CN,CN,CN,0,0,3,88.3991,216.735,...,3.35191,1.48509,6.67273,11.04440,2.19577,0.8,0.7,0.444681,0.6000,26.1
21,21,bl,CN,CN,CN,0,0,3,93.1445,211.345,...,5.46201,3.19832,56.56640,203.94900,153.71500,3.6,9.4,0.512884,0.4817,14.4


In [ ]:
progression_df = progression_df.sort_values(by='RID').reset_index(drop=True)

In [ ]:
progression_df.head(100)

,RID,VISCODE,DX_bl,BASELINE_DX,FINAL_DX,PROGRESSION_MONTH,PROGRESSED,TOTAL_VISITS,L_HISTIDINE,L_LYSINE,...,NORCA,NORDCA,TCA,TCDCA,TDCA,TUDCA,TAMCA,UCA,UDCA,BUDCA
0,3,bl,Dementia,Dementia,Dementia,0,0,3,99.4952,261.459,...,16.97580,2.935060,55.71210,154.11700,175.796000,1.7,11.3,0.818550,15.6,25.1
1,4,bl,LMCI,LMCI,LMCI,0,0,2,78.1877,208.347,...,4.12863,2.056650,15.22970,97.38330,26.792700,2.1,1.6,0.225801,24.5,216.8
2,5,bl,CN,CN,CN,0,0,3,95.3085,185.013,...,11.09640,1.300000,349.33300,501.46500,455.264000,11.1,13.3,1.239740,86.8,224.3
3,6,bl,LMCI,LMCI,LMCI,0,0,3,89.7745,297.523,...,3.43190,3.596810,16.82620,75.07010,13.342100,1.6,1.3,0.991451,13.1,19.1
4,7,bl,Dementia,Dementia,Dementia,0,0,3,115.9300,252.900,...,20.98920,4.039030,14.28970,24.61530,6.877590,1.1,2.0,23.490600,79.6,342.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,173,bl,CN,CN,MCI,96,1,4,91.6548,210.510,...,4.35901,3.345700,24.30960,62.28130,44.954500,0.5,3.8,0.529215,4.8,45.7
96,176,bl,LMCI,LMCI,LMCI,0,0,3,107.9890,192.918,...,10.73760,2.444620,14.41510,34.70320,16.001800,0.7,3.5,0.678146,40.3,28.6
97,177,bl,CN,CN,CN,0,0,5,92.4661,143.544,...,3.30936,0.600062,108.27200,245.51500,174.837000,1.6,22.4,0.258082,7.5,33.9
98,178,bl,LMCI,LMCI,LMCI,0,0,5,84.4765,214.314,...,1.66672,1.442890,3.70206,5.67148,12.300000,0.5,1.5,0.502634,2.8,104.3


In [ ]:
progression_df.to_csv('bl_progression_data.csv', index=False)